# MITGCM Intermediate Parameters Processing for regional Temporal Scale Analysis

**Purpose**: Code for computing intermediate derived variables from the model diagnostics for the temporal scale analysis. These include: 

1. Computing conservative temperature, Absolute Salinity, and potential density
2. Interpolate the horizontal velocity components

**Luke Colosi | lcolosi@ucsd.edu**

Force matplotlib plots to display directly within the output cell of the notebook

In [1]:
%matplotlib inline

Import python libraries

In [2]:
import sys
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt 
from netCDF4 import Dataset, num2date
from datetime import datetime
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import cmocean
from scipy.interpolate import interp1d
import gsw
from pyproj import Proj, Transformer

# Set path to access python functions
ROOT = '/Users/lukecolosi/Desktop/projects/graduate_research/Gille_lab/'
sys.path.append(ROOT + 'AirSeaCoupling/tools/')

#--- Other Functions ---# 
import cartopy_figs as cart

Set data analysis parameters

In [5]:
# Set processing parameters
option_proc          = 'ssh' # Speicifies which processing will occur. Options include: 'vel', 'density', or 'ssh'
option_depth         = 0.5     # Specifies the depth to preform data processing at
option_grad          = 0 

# Set path to project directory
PATH = ROOT + 'AirSeaCoupling/data/mitgcm/SWOT_MARA_RUN4_LY/spatial/regional/'

Load mitgcm data netcdf files 

In [6]:
#------------------------------------------# 
# Velocity Processing
#------------------------------------------# 
if option_proc == 'vel':

    # Obtain filename paths
    filename_u = PATH + "UVEL_CCS4_hrly_map_depth_" + str(option_depth) + "m.nc"
    filename_v = PATH + "VVEL_CCS4_hrly_map_depth_" + str(option_depth) + "m.nc"

    # Generate the nc data structure
    nc_u = Dataset(filename_u, 'r')
    nc_v = Dataset(filename_v, 'r')

    # Extract data variables
    depth  = nc_u.variables['Z'][:]
    lon_XG    = nc_u.variables['XG'][:]
    lat_YC    = nc_u.variables['YC'][:]
    lon_XC    = nc_v.variables['XC'][:]
    lat_YG    = nc_v.variables['YG'][:]
    time   =  num2date(nc_u.variables['time'][:], nc_u.variables['time'].units)

    u_raw  = nc_u.variables['UVEL'][:]
    v_raw  = nc_v.variables['VVEL'][:]

    # Mask data at fill values (zero for the MITgcm output)
    u_m = np.ma.masked_where(u_raw == 0, u_raw)
    v_m = np.ma.masked_where(v_raw == 0, v_raw)

#------------------------------------------# 
# Density Processing
#------------------------------------------# 
elif option_proc == 'density':

    # Obtain filename paths
    filename_temp = PATH + "THETA_CCS4_hrly_map_depth_" + str(option_depth) + "m.nc"
    filename_salt = PATH + "SALT_CCS4_hrly_map_depth_" + str(option_depth) + "m.nc"

    # Generate the nc data structure
    nc_temp = Dataset(filename_temp, 'r')
    nc_salt = Dataset(filename_salt, 'r')

    # Extract data variables
    depth = nc_temp['Z'][:]
    lon = nc_temp.variables['XC'][:]
    lat = nc_temp.variables['YC'][:]
    time =  num2date(nc_temp.variables['time'][:], nc_temp.variables['time'].units)

    T = nc_temp.variables['THETA'][:]
    S = nc_salt.variables['SALT'][:]

    # Mask data at fill values (zero for the MITgcm output)
    T_m = np.ma.masked_where(T == 0, T)
    S_m = np.ma.masked_where(S == 0, S)

#------------------------------------------# 
# SSH Processing
#------------------------------------------# 
elif option_proc == 'ssh':

    # Obtain filename paths
    filename_ssh = PATH + "ETAN_CCS4_hrly_map_surface.nc"

    # Generate the nc data structure
    nc_ssh = Dataset(filename_ssh, 'r')

    # Extract data variables
    lon = nc_ssh.variables['XC'][:]
    lat = nc_ssh.variables['YC'][:]
    time =  num2date(nc_ssh.variables['time'][:], nc_ssh.variables['time'].units)

    ssh = nc_ssh.variables['ETAN'][:]

    # Mask data at fill values (zero for the MITgcm output)
    ssh_m = np.ma.masked_where(ssh == 0, ssh)

# Convert cftime.DatetimeGregorian to Python datetime objects
time_dt = np.array([datetime(d.year, d.month, d.day, d.hour, d.minute, d.second) for d in time])

**Processing horizontal velocity**

In [5]:
if option_proc == 'vel':

    # Convert to a ndarray
    u_m = np.asarray(u_m)
    v_m = np.asarray(v_m)
    lon_XG = np.asarray(lon_XG)
    lat_YC = np.asarray(lat_YC)
    lon_XC = np.asarray(lon_XC)
    lat_YG = np.asarray(lat_YG)

    # Slice lon_XG and lat_YG to match lon_XC and lat_YC bounds respectively 
    lon_min, lon_max = np.min(lon_XC), np.max(lon_XC)
    lat_min, lat_max = np.min(lat_YC), np.max(lat_YC)
    idx_lon = (lon_XG >= lon_min) & (lon_XG <= lon_max)
    idx_lat = (lat_YG >= lat_min) & (lat_YG <= lat_max)
    lon_XG_c = lon_XG[idx_lon]
    lat_YG_c = lat_YG[idx_lat]

    # Apply the same slicing operation to u_raw and v_raw (recall: dim(u_raw) =  (time,lat_YC,lon_XG) and dim(v_raw) =  (time,lat_YG,lon_XC))
    u_raw_c = u_m[:,:,idx_lon]
    v_raw_c = v_m[:,idx_lat,:]

    # Set processing parameters
    ntime,_,_ = np.shape(u_raw_c)
    nlat,nlon = np.size(lat_YC),np.size(lon_XC)
    lon       = lon_XC
    lat       = lat_YC 

    # Initalize arrays
    u_int  = np.zeros((ntime,nlat,nlon)) 
    v_int  = np.zeros((ntime,nlat,nlon))

    if option_grad == 1: 
        vort   = np.zeros((ntime,nlat,nlon)) 
        div    = np.zeros((ntime,nlat,nlon)) 
        strain = np.zeros((ntime,nlat,nlon)) 
        speed  = np.zeros((ntime,nlat,nlon)) 
        dir    = np.zeros((ntime,nlat,nlon)) 
 

    # Loop through time
    for itime in range(0,ntime): 

        # Set progress bar
        progress = (itime + 1) / (len(time))
        sys.stdout.write(f"\rProgress: {progress:.1%}")
        sys.stdout.flush()

        # Grab the ith time frame 
        u_i = np.squeeze(u_raw_c[itime,:,:])
        v_i = np.squeeze(v_raw_c[itime,:,:])

        # Interpolate u_z from YC,XG grid onto the YC,XC grid 
        # Interpolate each row along columns (axis=1)
        u_int[itime,:,:] = np.array([
                            interp1d(lon_XG_c, row, kind='linear', bounds_error=False)(lon)
                            for row in u_i
        ])

        # Interpolate v_z from YG,XC grid onto the YC,XC grid 
        v_int[itime,:,:] = np.array([
                            interp1d(lat_YG_c, col, kind='linear', bounds_error=False)(lat)
                            for col in v_i.T
        ]).T 

        # Compute horizontal gradient statistics 
        if option_grad == 1: 
            
            # Compute the horizontal speed 
            speed[itime,:,:] = np.sqrt(u_int[itime,:,:]**2 + v_int[itime,:,:]**2)

            # Compute the direction (directional convention: Going towards, CW, reference north)
            dir[itime,:,:] = (90 - np.degrees(np.arctan2(v_int[itime,:,:], u_int[itime,:,:]))) % 360

            # Compute spatial derivatives at the scale of the grid
            dx = abs(lon[1] - lon[0]) * 111*10**3 # Units: meters (ignoring the spherical dependency)
            dy = abs(lat[1] - lat[0]) * 111*10**3 # Units: meters 
            dudx = np.gradient(u_int[itime,:,:], dx, axis=1) # Units: m/s/m = 1/s
            dvdy = np.gradient(v_int[itime,:,:], dy, axis=0)
            dvdx = np.gradient(v_int[itime,:,:], dx, axis=1)
            dudy = np.gradient(u_int[itime,:,:], dy, axis=0)

            # Compute the coriolis parameter (the local vertical planetary vorticity)
            omega = 7.2921 * 10**(-5)            # Magnitude of Earth's rotation rate, Units: rad/s 
            f = 2*omega*np.sin(np.deg2rad(lat))  # Units: rad/s

            # Compute vorticity, divergence, and strain normalized by the planetary vorticity
            vort[itime,:,:]   = (dvdx - dudy)/f[:, np.newaxis]
            div[itime,:,:]    = (dudx + dvdy)/f[:, np.newaxis]
            strain[itime,:,:] = (((dudx - dvdy)**2 + (dvdx + dudy)**2)**(1/2))/f[:, np.newaxis]  # Total strain magnitude linear combination of stretching and angular spreading strain

**Processing Conservative Temperature, Absolute Salinity, and Potential Density** 

In [6]:
if option_proc == 'density': 

    # Set the dimensions of the array
    ntime, nlat, nlon = T_m.shape

    # Compute pressure once for each latitude, since pressure is independent of time and longitude
    pressure_lat = np.array([gsw.p_from_z(depth, lat[i]) for i in range(nlat)])  

    # Broadcast pressure, lon, and lat to shape of full array
    pressure = np.broadcast_to(pressure_lat[None, :, None], (ntime, nlat, nlon))
    lon3d = np.broadcast_to(lon[None, None, :], (ntime, nlat, nlon))
    lat3d = np.broadcast_to(lat[None, :, None], (ntime, nlat, nlon))

    # Compute Absolute Salinity
    SA = gsw.SA_from_SP(S_m, pressure, lon3d, lat3d)

    # Compute Conservative Temperature
    CT = gsw.CT_from_pt(SA, T_m)

    # Compute in-situ density
    density = gsw.rho(SA, CT, pressure)

    # Compute potential density anomaly (sigma0)
    sigma0 = gsw.sigma0(SA, CT)

**Processing Sea Surface Height** (Compute geostrophic velocity)

In [8]:
if option_proc == 'ssh':

    # Set the dimensions of the array
    ntime, nlat, nlon = ssh_m.shape 

    

Save intermediate data in a netCDF for future use

In [20]:
# Save variables in data arrays

if option_proc == 'density': 

    #--- Coordinates ---# 
    Depth = xr.DataArray(data=depth, 
                        dims=(),
                        attrs=dict(
                            description='Depth level of sea-state variables.',
                            units='m'
                            )
    )

    #--- Sea State Varibles ---# 
    Pressure = xr.DataArray(data=pressure, 
                        dims=['time','lat','lon'],
                        coords=dict(time=time_dt,lat=lat,lon=lon),
                        attrs=dict(
                            description='Pressure regional map off point conception, CA.',
                            units='dbar'
                            )
    )

    Density = xr.DataArray(data=density, 
                        dims=['time','lat','lon'],
                        coords=dict(time=time_dt,lat=lat,lon=lon),
                        attrs=dict(
                            description='In-situ Density regional map off point conception, CA.',
                            units='kg/m^3'
                            )
    ) 

    SIG = xr.DataArray(data=sigma0, 
                        dims=['time','lat','lon'],
                        coords=dict(time=time_dt,lat=lat,lon=lon),
                        attrs=dict(
                            description='Potential Density anomaly regional map off point conception, CA referenced to the pressure at the sea surface.',
                            units='kg/m^3'
                            )
    ) 

    CTemp = xr.DataArray(data=CT, 
                        dims=['time','lat','lon'],
                        coords=dict(time=time_dt,lat=lat,lon=lon),
                        attrs=dict(
                            description='Conservative temperature regional map off point conception, CA.',
                            units='degrees Celcius'
                            )
    ) 

    ASal = xr.DataArray(data=SA, 
                        dims=['time','lat','lon'],
                        coords=dict(time=time_dt,lat=lat,lon=lon),
                        attrs=dict(
                            description='Absolute Salinity regional map off point conception, CA.',
                            units='g/kg'
                            )
    )

if option_proc == 'vel': 

    #--- Coordinates ---# 
    Depth = xr.DataArray(data=depth, 
                        dims=(),
                        attrs=dict(
                            description='Depth level of sea-state variables.',
                            units='m'
                            )
    )

    #--- Velocity Components ---#
    u = xr.DataArray(data=u_int,
                        dims=['time','lat','lon'],
                        coords=dict(time=time_dt,lat=lat,lon=lon),
                        attrs=dict(
                            description='The x-component (zonal) of velocity interpolated onto (XC,YC) grid.',
                            units='m/s'
                        )
    )

    v = xr.DataArray(data=v_int,
                        dims=['time','lat','lon'],
                        coords=dict(time=time_dt,lat=lat,lon=lon),
                        attrs=dict(
                            description='The y-component (meridional) of velocity interpolated onto (XC,YC) grid.',
                            units='m/s'
                        )
    )

if option_proc == 'ssh':

    #--- Sea Surface Height ---#
    ssh = xr.DataArray(data=ssh_m,
                        dims=['time','lat','lon'],
                        coords=dict(time=time_dt,lat=lat,lon=lon),
                        attrs=dict(
                            description='Sea surface height regional map off point conception, CA.',
                            units='m'
                        )
     )

# Create a data set from data arrays 
if option_proc == 'vel':  

    data = xr.Dataset({'Depth':Depth,'u':u,'v':v,})
    file_path = PATH + "/mitgcm_intermediate_data_vel_hrly_map_depth_" + str(option_depth) + "m.nc"

if option_proc == 'density':  

    data = xr.Dataset({'Depth':Depth,'Pressure':Pressure,'Density':Density,'SIG':SIG,'CTemp':CTemp,'ASal':ASal})
    file_path = PATH + "/mitgcm_intermediate_data_TSD_hrly_map_depth_" + str(option_depth) + "m.nc"

if option_proc == 'ssh':

    data = xr.Dataset({'ssh':ssh})
    file_path = PATH + "/mitgcm_intermediate_data_ssh_hrly_map_surface.nc"

# Check if file exists, then delete it
if os.path.exists(file_path):
    os.remove(file_path)

# Create netcdf file
data.to_netcdf(file_path,mode='w')